In [2]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Import functions from your modules
from data_preparation import prepare_interaction_data
from train_test_features import train_test_split_by_cell
from features_enginearing import (
    add_interaction_distance_and_strand_features,
    compile_chromatin_features,
    merge_conservation_with_interactions,
)
from n import computed_neighboring_features,data_neighboring_features

# === Paths ===
PATH_MOTIF_CHROMATIN_FEATURES = "D:/m.mirabolghasemi/New folder/Motifs1000"
PATH_INTERSECT_CHIAPET = "D:/m.mirabolghasemi/New folder/node_motif_rel/with_dup"
PATH_INTERSECT_CHIPSEQ = "D:/m.mirabolghasemi/New folder/CTCF_Motifs"
PATH_RPKM_FEATURES = "D:/m.mirabolghasemi/New folder/rpkm_output"
PATH_RPKM_INTERVALS = "D:/m.mirabolghasemi/New folder/features/features"
PATH_INTERSECT_MOTIF_CHIA = "D:/m.mirabolghasemi/New folder/overlaps1000max"
PATH_CTCF_MOTIFS = "D:/m.mirabolghasemi/New folder/CTCF_Motifs"

CHROMATIN_FEATURES = [
    "H3K4me1", "H3K4me2", "H3K4me3", "H3K9me3", "RAD21", "CTCF",
    "H3K36me3", "H3K79me2", "H3K27ac", "H3K9ac", "H3K27me3", "H2AFZ", "H4K20me1"
]

CELL_LINES = ["GM", "H1", "HCT116", "HepG2"]
TEST_CELL_LINE = "HCT116"  # Example test cell line


def prepare_all_features(
    chia_pet_dir: str,
    chipseq_dir: str,
    chromatin_features_dir: str,
    cell_lines: list[str],
    chromatin_features: list[str],
) -> pd.DataFrame:
    """
    Prepare the full dataset combining interaction, chromatin, and conservation features.

    Parameters
    ----------
    chia_pet_dir : str
        Path to directory with intersections between CTCF motifs and ChIA-PET data.
    chipseq_dir : str
        Path to directory with intersections between CTCF motifs and ChIP-Seq data.
    chromatin_features_dir : str
        Path to motif chromatin features directory.
    cell_lines : list[str]
        List of cell lines for which features should be combined.
    chromatin_features : list[str]
        List of chromatin features to include.

    Returns
    -------
    pd.DataFrame
        Combined DataFrame of all interaction, distance, strand,
        chromatin, and conservation features.
    """
    cell_lines_copy = list(cell_lines)

    print("Step 1: Preparing motif interaction data...")
    all_cells_interactions = prepare_interaction_data(
        chia_pet_dir, chipseq_dir, cell_lines_copy
    )

    print("Step 2: Adding interaction distance and strand features...")
    interaction_features_df = add_interaction_distance_and_strand_features(
        all_cells_interactions,
        chromatin_features_dir,
        cell_lines_copy,
    )

    print("Step 3: Compiling chromatin features...")
    chromatin_features_df = compile_chromatin_features(
        PATH_RPKM_FEATURES,
        PATH_RPKM_INTERVALS,
        chromatin_features_dir,
        cell_lines_copy,
        chromatin_features,
    )

    print("Step 4: Merging conservation features...")
    conservation_features_df = merge_conservation_with_interactions(
        all_cells_interactions,
        PATH_INTERSECT_MOTIF_CHIA,
        PATH_CTCF_MOTIFS,
        cell_lines_copy,
    )

    print("Step 5: Combining all features into one dataset...")
    all_features_df = (
        conservation_features_df
        .merge(
            interaction_features_df,
            on=["sequence_name", "start1", "stop1", "start2", "stop2"],
        )
        .merge(
            chromatin_features_df,
            on=["sequence_name", "start1", "stop1", "start2", "stop2"],
        )
    )

    print("✅ Feature preparation complete.")
    return all_features_df


def train_and_evaluate_model(train_df: pd.DataFrame, test_df: pd.DataFrame) -> pd.DataFrame:
    """
    Train a Gradient Boosting model and evaluate performance on the test set.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training data with features and labels.
    test_df : pd.DataFrame
        Test data with features and labels.

    Returns
    -------
    pd.DataFrame
        Test dataframe with predicted probabilities and labels.
    """
    print("Training Gradient Boosting Classifier...")

    nonpredictors = ["sequence_name", "start1", "stop1", "start2", "stop2", "label", "cell_line"]

    model = GradientBoostingClassifier(
        n_estimators=4000,
        learning_rate=0.1,
        max_depth=5,
        random_state=0,
    )

    X_train = train_df.drop(columns=nonpredictors)
    y_train = train_df["label"]

    X_test = test_df.drop(columns=nonpredictors)
    y_test = test_df["label"]

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, predictions)
    print(f"✅ Accuracy: {accuracy:.4f}")

    cm = confusion_matrix(y_test, predictions, labels=model.classes_)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
    disp.plot()

    test_df["probability"] = probabilities
    test_df["prediction"] = predictions

    return test_df





In [ ]:
# === Main pipeline ===
if __name__ == "__main__":
    pairs_df = prepare_all_features(
        PATH_INTERSECT_CHIAPET,
        PATH_INTERSECT_CHIPSEQ,
        PATH_MOTIF_CHROMATIN_FEATURES,
        CELL_LINES,
        CHROMATIN_FEATURES,
    )

    # train/test split (keeps CELL_LINES unchanged)
    train_df, test_df = train_test_split_by_cell(
        pairs_df,
        CELL_LINES,
        TEST_CELL_LINE,
        CHROMATIN_FEATURES,
    )

    # run model
    train_neighboring_df=data_neighboring_features(train_df,3)
test_neighboring_df=data_neighboring_features(test_df,3)
    test_confidence_df = train_and_evaluate_model(train_df, test_df)
   
    

Step 1: Preparing motif interaction data...
Step 2: Adding interaction distance and strand features...


e:\Paper_Codes\features_enginearing.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged["nodes_pos"] = merged.apply(get_strand_orientation, axis=1)


Step 3: Compiling chromatin features...
Step 4: Merging conservation features...
Step 5: Combining all features into one dataset...
✅ Feature preparation complete.
